In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports done")

In [ ]:
# Cell 2 — Load data
df = pd.read_csv("../data/raw/github_features.csv")

X = df.drop(columns=["username", "churned"])
y = df["churned"]

print("Shape:", X.shape)
print("Churn rate:", y.mean().round(3))
print(X.head())

In [ ]:
# Cell 3 — Class balance plot
y.value_counts().plot(kind="bar", color=["steelblue", "tomato"])
plt.title("Churn Class Balance")
plt.xticks([0, 1], ["Not Churned", "Churned"], rotation=0)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../data/raw/class_balance.png")
plt.show()
print("✅ Class balance looks good")

In [ ]:
# Cell 4 — Correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../data/raw/correlation_matrix.png")
plt.show()

In [ ]:
# Cell 5 — METHOD 1: Filter Methods
print("=" * 50)
print("METHOD 1 — FILTER METHODS")
print("=" * 50)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Variance Threshold
sel = VarianceThreshold(threshold=0.01)
sel.fit(X_scaled)
low_variance = X.columns[~sel.get_support()].tolist()
print(f"\nLow variance features (dropped): {low_variance}")

# ANOVA F-test
selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X_scaled, y)

filter_scores = pd.DataFrame({
    "feature": X.columns,
    "f_score": selector.scores_,
    "p_value": selector.pvalues_
}).sort_values("f_score", ascending=False)

print("\nANOVA F-test scores:")
print(filter_scores.to_string(index=False))

# Plot
plt.figure(figsize=(10, 5))
plt.barh(filter_scores["feature"], filter_scores["f_score"], color="steelblue")
plt.title("Filter Method — ANOVA F-scores")
plt.xlabel("F-score")
plt.tight_layout()
plt.savefig("../data/raw/filter_scores.png")
plt.show()


In [ ]:
# Cell 6 — METHOD 2: Wrapper RFE
print("=" * 50)
print("METHOD 2 — WRAPPER: RFE")
print("=" * 50)

model_rfe = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=model_rfe, n_features_to_select=5)
rfe.fit(X_scaled, y)

rfe_results = pd.DataFrame({
    "feature": X.columns,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
}).sort_values("ranking")

print(rfe_results.to_string(index=False))
print(f"\nRFE selected: {X.columns[rfe.support_].tolist()}")

In [ ]:
# Cell 7 — METHOD 3: Decision Tree
print("=" * 50)
print("METHOD 3 — DECISION TREE IMPORTANCE")
print("=" * 50)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_scaled, y)

dt_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": dt.feature_importances_
}).sort_values("importance", ascending=False)

print(dt_importance.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(dt_importance["feature"], dt_importance["importance"], color="orange")
plt.title("Decision Tree — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("../data/raw/dt_importance.png")
plt.show()

In [ ]:
# Cell 8 — METHOD 4: Random Forest
print("=" * 50)
print("METHOD 4 — RANDOM FOREST IMPORTANCE")
print("=" * 50)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(rf_importance.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(rf_importance["feature"], rf_importance["importance"], color="green")
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("../data/raw/rf_importance.png")
plt.show()

In [ ]:
# Cell 9 — Comparison Table
print("=" * 50)
print("FEATURE SELECTION COMPARISON TABLE")
print("=" * 50)

# Build ranks for each method
filter_ranks = filter_scores.reset_index(drop=True)
filter_ranks["filter_rank"] = filter_ranks.index + 1
filter_ranks = filter_ranks[["feature", "filter_rank"]]

rfe_df = rfe_results[["feature", "selected"]].copy()
rfe_df["rfe_selected"] = rfe_df["selected"].map({True: "✅", False: "❌"})
rfe_df = rfe_df[["feature", "rfe_selected"]]

dt_ranks = dt_importance.reset_index(drop=True)
dt_ranks["dt_rank"] = dt_ranks.index + 1
dt_ranks = dt_ranks[["feature", "dt_rank"]]

rf_ranks = rf_importance.reset_index(drop=True)
rf_ranks["rf_rank"] = rf_ranks.index + 1
rf_ranks = rf_ranks[["feature", "rf_rank"]]

comparison = filter_ranks \
    .merge(rfe_df, on="feature") \
    .merge(dt_ranks, on="feature") \
    .merge(rf_ranks, on="feature") \
    .sort_values("rf_rank")

print(comparison.to_string(index=False))
comparison.to_csv("../data/raw/feature_comparison.csv", index=False)
print("\n✅ Comparison table saved!")

In [ ]:
# Cell 10 — Final model with cross validation
print("=" * 50)
print("FINAL MODEL — CROSS VALIDATION")
print("=" * 50)

# Use top 5 features from RF
top_features = rf_importance.head(5)["feature"].tolist()
print(f"Selected features: {top_features}")

X_final = df[top_features]
X_final_scaled = scaler.fit_transform(X_final)

final_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

cv_results = cross_validate(
    final_model, X_final_scaled, y,
    cv=5,
    scoring=["accuracy", "precision", "recall", "f1"]
)

print(f"\nAccuracy:  {cv_results['test_accuracy'].mean():.3f}")
print(f"Precision: {cv_results['test_precision'].mean():.3f}")
print(f"Recall:    {cv_results['test_recall'].mean():.3f}")
print(f"F1 Score:  {cv_results['test_f1'].mean():.3f}")

In [ ]:
# Cell 11 — Save final model
import joblib

final_model.fit(X_final_scaled, y)
joblib.dump(final_model, "../app/model.pkl")
joblib.dump(scaler, "../app/scaler.pkl")
joblib.dump(top_features, "../app/features_list.pkl")

print(f"✅ Model saved to app/model.pkl")
print(f"✅ Scaler saved to app/scaler.pkl")
print(f"✅ Features list saved to app/features_list.pkl")
print(f"Top features: {top_features}")